# Faruq-v3 — AF2-family private Kaggle core updater

Membangun/update private Kaggle Dataset `faruq-v3-experiment-core-v1` khusus untuk eksperimen AF2-family saat ini:

- Stage 1: `AF2FS` vs `AF2FFAB2FS` (rFFT)
- Stage 2: `AF2FFAB2FS` vs `AF2FFADCTFS` (DCT), hanya jika Stage 1 PASS

Tidak ada STB/top-controls. Notebook ini tidak menjalankan training dan tidak memasukkan locked test.

Sebelum Run all, buat Colab secrets dan aktifkan notebook access:
`KAGGLE_USERNAME` dan `KAGGLE_API_TOKEN`.


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=True)

import importlib, json, os, shutil, subprocess, sys, time
from pathlib import Path

BRANCH='codex/af2-ffab2-from-start-dct'
REPO=Path('/content/coffee-bean-detection')

os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)

clone=['git','clone','--depth','1','--branch',BRANCH,
       'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(1,4):
    result=subprocess.run(clone)
    if result.returncode==0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt==3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)

subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade',
                'ultralytics==8.4.96','kaggle==2.2.2'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)

for name in list(sys.modules):
    if name=='coffee_detector' or name.startswith('coffee_detector.'):
        sys.modules.pop(name,None)
sys.path.insert(0,str(REPO/'src'))
importlib.invalidate_caches()
os.chdir(REPO)

print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
print('ULTRALYTICS:',__import__('ultralytics').__version__)


In [ ]:
from coffee_detector.drive_project import resolve_drive_project_root
from coffee_detector.experiments.prepare_af2_spectral_kaggle import (
    PROJECT_ARTIFACTS,
    build_af2_spectral_kaggle_bundle,
    MANIFEST_NAME,
    MANIFEST_FORMAT,
    D0_NAMES,
)

required=tuple(PROJECT_ARTIFACTS.values())
PROJECT=resolve_drive_project_root(required_relative_paths=required)
BUNDLE=Path('/content/faruq-v3-experiment-core-v1')

if BUNDLE.exists():
    shutil.rmtree(BUNDLE)

manifest=build_af2_spectral_kaggle_bundle(PROJECT,BUNDLE)

assert manifest['format']==MANIFEST_FORMAT
assert manifest['test_images_included'] is False
assert (BUNDLE/MANIFEST_NAME).is_file()
for name in PROJECT_ARTIFACTS:
    if not (BUNDLE/name).is_file():
        raise FileNotFoundError(BUNDLE/name)
for name in D0_NAMES:
    proof=manifest['checkpoint_validation'][name]
    assert proof['loadable_by_ultralytics'] is True
    assert int(proof['nc'])==21

print('PROJECT:',PROJECT)
print('BUNDLE :',BUNDLE)
print('MANIFEST:',BUNDLE/MANIFEST_NAME)
print('FILES:')
for p in sorted(BUNDLE.iterdir()):
    print(' ',p.name,p.stat().st_size)
print('AF2-FAMILY CORE BUILD: PASS')


In [ ]:
username=userdata.get('KAGGLE_USERNAME')
token=userdata.get('KAGGLE_API_TOKEN')
assert username and token, 'Aktifkan secret KAGGLE_USERNAME dan KAGGLE_API_TOKEN.'
username=username.strip()
assert username and '/' not in username, 'KAGGLE_USERNAME tidak valid.'

os.environ['KAGGLE_USERNAME']=username
os.environ['KAGGLE_API_TOKEN']=token
os.environ['KAGGLE_KEY']=token

dataset_id=f'{username}/faruq-v3-experiment-core-v1'
metadata={
    'title':'Faruq V3 Experiment Core V1',
    'id':dataset_id,
    'licenses':[{'name':'other'}],
    'isPrivate':True,
}
(BUNDLE/'dataset-metadata.json').write_text(json.dumps(metadata,indent=2)+'\n',encoding='utf-8')

probe=subprocess.run(
    ['kaggle','datasets','list','--mine','--search','faruq-v3-experiment-core-v1','--csv'],
    text=True,capture_output=True
)
if probe.returncode!=0:
    print(probe.stdout)
    print(probe.stderr)
    raise RuntimeError('Autentikasi/probe Kaggle gagal sebelum upload.')

exists=dataset_id.lower() in probe.stdout.lower()
message='AF2-family core: grouped development + D0 seeds 42/123/2026 + AF2 spectral contract'

if exists:
    print('Dataset ada; upload VERSION baru...',flush=True)
    upload=subprocess.run(
        ['kaggle','datasets','version','-p',str(BUNDLE),'-m',message,'--keep-tabular'],
        check=False
    )
    action='UPDATED'
else:
    print('Dataset belum ada; CREATE...',flush=True)
    upload=subprocess.run(
        ['kaggle','datasets','create','-p',str(BUNDLE),'--keep-tabular'],
        check=False
    )
    action='CREATED'

if upload.returncode!=0:
    raise RuntimeError(f'Kaggle {action.lower()} gagal: returncode={upload.returncode}')

print('UPLOAD:',action)
print('DATASET:',dataset_id)
print('PRIVATE: True | TEST: False')


In [ ]:
required=(
    'faruq-development-v3-grouped.tar.bin',
    'af2_spectral_kaggle_manifest.json',
    'D0_seed42_best.pt',
    'D0_seed123_best.pt',
    'D0_seed2026_best.pt',
    'lfdet_afab_seed42_screening.json',
    'af2_igem_paired_confirmation.json',
)

listing=''
for attempt in range(18):
    check=subprocess.run(
        ['kaggle','datasets','files',dataset_id,'--page-size','100','--csv'],
        text=True,capture_output=True
    )
    listing=check.stdout
    if check.returncode==0 and all(name in listing for name in required):
        break
    print(f'Menunggu indeks Kaggle: {attempt+1}/18',flush=True)
    time.sleep(10)
else:
    print(listing)
    print(check.stderr)
    raise RuntimeError('Upload diterima tetapi file AF2-family belum lengkap di indeks Kaggle.')

print('VERIFIKASI KAGGLE: PASS')
for name in required:
    print(' OK',name)
print()
print('NEXT:')
print('1) Kembali ke notebook Kaggle Stage 1.')
print('2) Remove input faruq-v3-experiment-core-v1 lama jika masih ter-cache.')
print('3) Add Input lagi versi terbaru.')
print('4) Jalankan AF2FS vs AF2FFAB2FS.')
print('5) DCT hanya setelah keputusan Stage 1 PASS.')
